# MS MARCO v1.1 - Hybrid Index Build

In [2]:
!pip install -q datasets pandas numpy beautifulsoup4 sentence-transformers faiss-cpu pyarrow tqdm nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.8 MB/s eta 0:00:00


In [3]:
import os
import re
import html
import json
import hashlib
import unicodedata
from collections import Counter, defaultdict

import nltk
import numpy as np
import pandas as pd
import faiss

from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
DATASET_NAME = "microsoft/ms_marco"
DATASET_CONFIG = "v1.1"
SPLITS = ["train", "validation", "test"]

OUTPUT_DIR = "outputs"
CORPUS_DIR = os.path.join(OUTPUT_DIR, "corpus")
INDEX_DIR = os.path.join(OUTPUT_DIR, "indexes")
BM25_DIR = os.path.join(INDEX_DIR, "bm25")
EMBEDDING_DIR = os.path.join(OUTPUT_DIR, "embeddings")

for directory in [CORPUS_DIR, INDEX_DIR, BM25_DIR, EMBEDDING_DIR]:
    os.makedirs(directory, exist_ok=True)

SPLIT_LIMITS = {
    "train": None,
    "validation": None,
    "test": None,
}

MAX_WORDS_PER_CHUNK = 256
OVERLAP_SENTENCES = 1

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_BATCH_SIZE = 128

In [5]:
def clean_text(text):
    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^\w\s.,;:!?()\-/%$'\"]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_for_id(text):
    text = clean_text(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def make_chunk_id(text):
    normalized = normalize_for_id(text)
    return hashlib.md5(normalized.encode("utf-8")).hexdigest()


def split_sentences(text):
    text = clean_text(text)
    if not text:
        return []
    return sent_tokenize(text)


def chunk_by_sentences(text, max_words=256, overlap_sentences=1):
    sentences = split_sentences(text)
    if not sentences:
        return []

    chunks = []
    current = []
    current_words = 0

    for sentence in sentences:
        sentence_words = len(sentence.split())

        if current and current_words + sentence_words > max_words:
            chunks.append(" ".join(current).strip())

            if overlap_sentences > 0:
                current = current[-overlap_sentences:]
                current_words = sum(len(s.split()) for s in current)
            else:
                current = []
                current_words = 0

        current.append(sentence)
        current_words += sentence_words

    if current:
        chunks.append(" ".join(current).strip())

    return [chunk for chunk in chunks if chunk]


def tokenize_for_bm25(text):
    text = clean_text(text).lower()
    return re.findall(r"\b\w+\b", text)

In [6]:
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG)
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

v1.1/validation-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

v1.1/train-00000-of-00001.parquet:   0%|          | 0.00/175M [00:00<?, ?B/s]

v1.1/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10047 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/82326 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9650 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 10047
    })
    train: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 82326
    })
    test: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 9650
    })
})

In [7]:
records_by_id = {}

for split in SPLITS:
    split_ds = dataset[split]
    limit = SPLIT_LIMITS.get(split)

    if limit is not None:
        split_ds = split_ds.select(range(min(limit, len(split_ds))))

    for row in tqdm(split_ds, desc=f"Collecting passages from {split}"):
        passages = row["passages"]
        passage_texts = passages.get("passage_text", [])
        urls = passages.get("url", [])

        for passage_idx, passage_text in enumerate(passage_texts):
            url = urls[passage_idx] if passage_idx < len(urls) else None
            chunks = chunk_by_sentences(
                passage_text,
                max_words=MAX_WORDS_PER_CHUNK,
                overlap_sentences=OVERLAP_SENTENCES,
            )

            for chunk in chunks:
                chunk = clean_text(chunk)
                if not chunk:
                    continue

                chunk_id = make_chunk_id(chunk)

                if chunk_id not in records_by_id:
                    records_by_id[chunk_id] = {
                        "chunk_id": chunk_id,
                        "text": chunk,
                        "url": url,
                    }

corpus_df = pd.DataFrame(records_by_id.values()).reset_index(drop=True)
corpus_df["row_id"] = np.arange(len(corpus_df), dtype=np.int64)
corpus_df = corpus_df[["row_id", "chunk_id", "text", "url"]]

corpus_path = os.path.join(CORPUS_DIR, "corpus.parquet")
corpus_df.to_parquet(corpus_path, index=False)

print("Corpus shape:", corpus_df.shape)
print("Saved:", corpus_path)
corpus_df.head()

Corpus shape: (757997, 4)
Saved: outputs/corpus/corpus.parquet


,row_id,chunk_id,text,url
0,0,e3f9cd0d6c321dceb17ed3cde55e7844,"Since 2007, the RBA's outstanding reputation h...",https://en.wikipedia.org/wiki/Reserve_Bank_of_...
1,1,6b1fee21eb83f0e809b255856ad9def0,The Reserve Bank of Australia (RBA) came into ...,https://en.wikipedia.org/wiki/Reserve_Bank_of_...
2,2,7273dbcf0ab20f894a7ff4034cb2ee82,RBA Recognized with the 2014 Microsoft US Regi...,http://acronyms.thefreedictionary.com/RBA
3,3,e511f5cab0ebd2de8b5687b285f25c07,The inner workings of a rebuildable atomizer a...,https://www.slimvapepen.com/rebuildable-atomiz...
4,4,e08e2d0d9123544a060416d49bfd31de,Results-Based Accountability (also known as RB...,http://rba-africa.com/about/what-is-rba/


In [8]:
vocab = {}
postings = defaultdict(list)
doc_len = np.zeros(len(corpus_df), dtype=np.int32)

for row_id, text in tqdm(enumerate(corpus_df["text"]), total=len(corpus_df), desc="Building BM25 raw stats"):
    tokens = tokenize_for_bm25(text)
    doc_len[row_id] = len(tokens)

    tf_counts = Counter(tokens)
    for token, tf_value in tf_counts.items():
        term_id = vocab.get(token)
        if term_id is None:
            term_id = len(vocab)
            vocab[token] = term_id

        postings[term_id].append((row_id, tf_value))

num_docs = len(corpus_df)
num_terms = len(vocab)
avgdl = float(doc_len.mean()) if num_docs > 0 else 0.0

df = np.zeros(num_terms, dtype=np.int32)
indptr = np.zeros(num_terms + 1, dtype=np.int64)

posting_doc_ids = []
posting_tf = []

for term_id in tqdm(range(num_terms), desc="Packing inverted index"):
    term_postings = postings[term_id]
    df[term_id] = len(term_postings)
    indptr[term_id + 1] = indptr[term_id] + len(term_postings)

    for doc_id, tf_value in term_postings:
        posting_doc_ids.append(doc_id)
        posting_tf.append(tf_value)

posting_doc_ids = np.asarray(posting_doc_ids, dtype=np.int32)
posting_tf = np.asarray(posting_tf, dtype=np.int16)

with open(os.path.join(BM25_DIR, "vocab.json"), "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)

np.save(os.path.join(BM25_DIR, "df.npy"), df)
np.save(os.path.join(BM25_DIR, "doc_len.npy"), doc_len)
np.save(os.path.join(BM25_DIR, "indptr.npy"), indptr)
np.save(os.path.join(BM25_DIR, "posting_doc_ids.npy"), posting_doc_ids)
np.save(os.path.join(BM25_DIR, "posting_tf.npy"), posting_tf)
np.save(os.path.join(BM25_DIR, "avgdl.npy"), np.array(avgdl, dtype=np.float32))

metadata = {
    "num_docs": int(num_docs),
    "num_terms": int(num_terms),
    "avgdl": avgdl,
    "total_postings": int(len(posting_doc_ids)),
    "doc_id_type": "row_id in corpus.parquet",
    "formula": "idf * tf*(k1+1)/(tf + k1*(1-b+b*dl/avgdl))",
    "idf_formula": "log(1 + (N - df + 0.5)/(df + 0.5))",
}

with open(os.path.join(BM25_DIR, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("BM25 raw stats saved to:", BM25_DIR)
print(metadata)

Building BM25 raw stats:   0%|          | 0/757997 [00:00<?, ?it/s]

Packing inverted index:   0%|          | 0/331509 [00:00<?, ?it/s]

BM25 raw stats saved to: outputs/indexes/bm25
{'num_docs': 757997, 'num_terms': 331509, 'avgdl': 73.06902006208468, 'total_postings': 37740092, 'doc_id_type': 'row_id in corpus.parquet', 'formula': 'idf * tf*(k1+1)/(tf + k1*(1-b+b*dl/avgdl))', 'idf_formula': 'log(1 + (N - df + 0.5)/(df + 0.5))'}


In [9]:
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = corpus_df["text"].tolist()
embeddings = model.encode(
    texts,
    batch_size=EMBEDDING_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

embeddings_path = os.path.join(EMBEDDING_DIR, "corpus_embeddings.npy")
np.save(embeddings_path, embeddings)

print("Embeddings shape:", embeddings.shape)
print("Embeddings saved:", embeddings_path)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/5922 [00:00<?, ?it/s]

Embeddings shape: (757997, 384)
Embeddings saved: outputs/embeddings/corpus_embeddings.npy


In [10]:
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings)

faiss_path = os.path.join(INDEX_DIR, "faiss_index.index")
faiss.write_index(faiss_index, faiss_path)

print("FAISS index saved:", faiss_path)
print("Total vectors:", faiss_index.ntotal)
print("Dimension:", dimension)

FAISS index saved: outputs/indexes/faiss_index.index
Total vectors: 757997
Dimension: 384


In [11]:
print("Done.")
print("Corpus:", os.path.join(CORPUS_DIR, "corpus.parquet"))
print("BM25 directory:", BM25_DIR)
print("Embeddings:", os.path.join(EMBEDDING_DIR, "corpus_embeddings.npy"))
print("FAISS:", os.path.join(INDEX_DIR, "faiss_index.index"))

Done.
Corpus: outputs/corpus/corpus.parquet
BM25 directory: outputs/indexes/bm25
Embeddings: outputs/embeddings/corpus_embeddings.npy
FAISS: outputs/indexes/faiss_index.index


In [12]:
import os
import re
import json
import numpy as np
import pandas as pd

BM25_DIR = "outputs/indexes/bm25"
CORPUS_PATH = "outputs/corpus/corpus.parquet"

corpus_df = pd.read_parquet(CORPUS_PATH)

with open(os.path.join(BM25_DIR, "vocab.json"), encoding="utf-8") as f:
    vocab = json.load(f)

with open(os.path.join(BM25_DIR, "metadata.json"), encoding="utf-8") as f:
    bm25_meta = json.load(f)

df = np.load(os.path.join(BM25_DIR, "df.npy"))
doc_len = np.load(os.path.join(BM25_DIR, "doc_len.npy"))
indptr = np.load(os.path.join(BM25_DIR, "indptr.npy"))
posting_doc_ids = np.load(os.path.join(BM25_DIR, "posting_doc_ids.npy"))
posting_tf = np.load(os.path.join(BM25_DIR, "posting_tf.npy"))

N = bm25_meta["num_docs"]
avgdl = bm25_meta["avgdl"]


def bm25_tokenize(text):
    text = str(text).lower()
    return re.findall(r"\b\w+\b", text)


def bm25_search(query, top_k=10, k1=1.5, b=0.75):
    scores = np.zeros(N, dtype=np.float32)

    for token in bm25_tokenize(query):
        term_id = vocab.get(token)
        if term_id is None:
            continue

        start = indptr[term_id]
        end = indptr[term_id + 1]

        docs = posting_doc_ids[start:end]
        tf = posting_tf[start:end].astype(np.float32)
        term_df = df[term_id]

        idf = np.log(1.0 + (N - term_df + 0.5) / (term_df + 0.5))
        denom = tf + k1 * (1.0 - b + b * doc_len[docs] / avgdl)
        scores[docs] += idf * (tf * (k1 + 1.0) / denom)

    nonzero = np.flatnonzero(scores)
    if len(nonzero) == 0:
        return corpus_df.iloc[[]].copy()

    top_idx = nonzero[np.argsort(scores[nonzero])[-top_k:][::-1]]
    results = corpus_df.iloc[top_idx].copy()
    results["bm25_score"] = scores[top_idx]
    return results[["bm25_score", "row_id", "chunk_id", "text", "url"]]


bm25_search("what is a corporation", top_k=5)

,bm25_score,row_id,chunk_id,text,url
79873,14.585622,79873,b2c63ba70b5c7193459b568031956057,In another kind of corporation the legal docum...,https://en.wikipedia.org/wiki/Corporation
155239,14.307276,155239,39dfce9860616ae306a01364ef2fdd6e,Characteristics of Transnational Corporations....,http://study.com/academy/lesson/transnational-...
456621,14.100653,456621,592de31d63c9f82a35d0fcaa66f8f1a4,What is Rational Unified Process (RUP). The co...,http://www.ibuzzle.com/articles/what-is-ration...
681695,13.415644,681695,153c7068be2ae2a6e8f2b2d500700383,"From Wikipedia, the free encyclopedia. In acco...",https://en.wikipedia.org/wiki/Minority_interest
79489,13.296777,79489,9e891e77bcba8f4ce6f251def25def97,Check what your PC is running. namebench-1.3.1...,http://www.processchecker.com/file/namebench-1...


In [13]:
!zip -r outputs.zip outputs

  adding: outputs/ (stored 0%)
  adding: outputs/corpus/ (stored 0%)
  adding: outputs/corpus/corpus.parquet (deflated 15%)
  adding: outputs/embeddings/ (stored 0%)
  adding: outputs/embeddings/corpus_embeddings.npy (deflated 7%)
  adding: outputs/indexes/ (stored 0%)
  adding: outputs/indexes/faiss_index.index (deflated 7%)
  adding: outputs/indexes/bm25/ (stored 0%)
  adding: outputs/indexes/bm25/posting_tf.npy (deflated 89%)
  adding: outputs/indexes/bm25/df.npy (deflated 82%)
  adding: outputs/indexes/bm25/metadata.json (deflated 29%)
  adding: outputs/indexes/bm25/vocab.json (deflated 60%)
  adding: outputs/indexes/bm25/indptr.npy (deflated 80%)
  adding: outputs/indexes/bm25/avgdl.npy (deflated 47%)
  adding: outputs/indexes/bm25/doc_len.npy (deflated 68%)
  adding: outputs/indexes/bm25/posting_doc_ids.npy (deflated 53%)
